# Contrastive Probe Inference

Runs the spatial-grounding probe as a factorial of control conditions over two
scene sources, logging both the executable action and a continuous readout.

**Continuous readout.** OpenVLA emits one token per action dimension and decodes
it to a bin centre, so the executable action is quantised. The lateral bin is
roughly 1e-3 wide while observed lateral predictions sit between 1e-4 and 4e-3
of zero, which put the earlier median paired difference at exactly zero in every
stratum with several pairs bit-identical between the two instructions. Alongside
the argmax action, each prediction now records the expected bin centre under the
model's own distribution over action tokens, which resolves differences smaller
than one bin. 

**Control conditions.** A model mapping the token `left` to a leftward action without consulting the
image reproduces the expected sign flip exactly, and a model whose lateral output
ignores the image produces no difference for reasons unrelated to language. Each
condition varies one factor: mirroring reverses the lateral axis with the
instruction held fixed, pairing an instruction with another scene removes its
referent while leaving the language intact, and removing the spatial term gives a
within-scene reference.

**Two scene sources.** The experiments run on the `constructed` scenes, which hold
two instances of the target noun and whose geometry is recorded rather than
inferred. The `bridge` scenes are the unaltered frames harvested into the validation
role, and they answer whether a
constructed-set finding transfers to data that was never edited. The two sets are
disjoint by construction, so no frame appears in both.

The constructed source is read from the frozen `evaluation_set.csv` written by
Notebook 03, not from the constructed manifest. The manifest holds everything ever
built, including scenes that failed the approval screen and the overbuild beyond the
target, and probing those would spend GPU time on stimuli the analysis excludes
while making the set that was measured depend on when the probe happened to run.

Output is `probe_predictions.csv` under the v2 generation, written fresh. Earlier
logs are not migrated: they were collected over a different scene set under a
different design, and copying them forward would mix two populations in one file.

## 0. Dependencies

OpenVLA-7B loads only against the pinned dependency set, which is installed here
rather than inherited from a session that happened to run Notebook 01 first. A
runtime that has not installed it keeps the Colab defaults, whose transformers 5.x
releases no longer expose `AutoModelForVision2Seq`, and the mismatch surfaces as an
import error inside [model.py](model.py) in the section below.

The pins are restated in this notebook, mirroring `requirements-colab.txt` and
Notebook 01, because they have to be installed before the repository is cloned and
before any pinned package is imported. A module already imported keeps its own code
for the life of the interpreter, so installing first is what makes the replacement
effective.

Three properties of the install matter:

- The interpreter version is asserted before anything else, because it decides
  whether the pinned set can be installed at all. `tokenizers==0.19.1`, which
  OpenVLA's remote modelling code requires, publishes no wheel beyond CPython 3.12.
  Pin the Colab runtime version to 2026.07 under Runtime > Change runtime type.
- `--only-binary=:all:` forbids a source build, so a missing wheel stops the install
  instead of failing late inside a compiler and leaving the pre-installed versions
  in place.
- torch is never reinstalled. The Colab build is matched to its CUDA driver, and
  overriding it tends to break GPU support.

The verification that closes the cell reads both the versions recorded on disk and
the version of any pinned module already imported. Those disagree when a package was
imported before the install, which is the one condition a session restart resolves,
and it is named here rather than left to surface as an obscure failure inside the
model load.

`pip` may report dependency conflicts against pre-installed Colab packages such as
`datasets` and `diffusers`, which want a newer `huggingface_hub`. Those packages are
not used anywhere in this pipeline and the conflict is expected.

In [12]:
import subprocess
import sys
from importlib.metadata import version

REQUIRED_PYTHON = (3, 12)          # highest version with wheels for the pinned set
COLAB_RUNTIME_VERSION = '2026.07'  # last runtime version shipping that interpreter

# Mirrors requirements-colab.txt and the install cell of Notebook 01, restated here
# because this cell runs before the repository is cloned. The trio at the top is the
# OpenVLA authors' known-good set; newer releases cause a misleading "requires
# prismatic" load error, and the 5.x line withdraws the interface model.py loads
# through. protobuf is bounded above as well as below, mirroring data.PROTOBUF_SPEC,
# and installed without --upgrade so a conforming runtime is left in place.
PINS = [
    'transformers==4.40.1',
    'tokenizers==0.19.1',
    'timm==0.9.10',
    'huggingface_hub==0.23.4',
    'accelerate==0.30.1',
    'bitsandbytes>=0.45.0',
    'protobuf>=6.31.1,<7',
]
EXPECTED = {
    'transformers': '4.40.1',
    'tokenizers': '0.19.1',
    'timm': '0.9.10',
    'huggingface_hub': '0.23.4',
    'accelerate': '0.30.1',
}

assert sys.version_info[:2] == REQUIRED_PYTHON, (
    f'Python {sys.version_info.major}.{sys.version_info.minor} is active, but the pinned '
    f'dependencies require Python {REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}. Set Runtime > '
    f'Change runtime type > Runtime version to {COLAB_RUNTIME_VERSION}, then reconnect and '
    f'run this notebook from the top.'
)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:', *PINS],
    check=True,
)

installed = {name: version(name) for name in EXPECTED}
for name, found in installed.items():
    print(f'{name:16s} {found}')

mismatched = {n: v for n, v in installed.items() if v != EXPECTED[n]}
assert not mismatched, (
    'Installed versions differ from the pin: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in mismatched.items())
    + '. Re-run this cell and check the install reported no error.'
)

# A module imported before the install still holds the previous code, which no
# further install can replace in this interpreter.
imported = {n: getattr(sys.modules[n], '__version__', '') for n in EXPECTED
            if n in sys.modules}
stale = {n: v for n, v in imported.items() if v and v != EXPECTED[n]}
assert not stale, (
    'These packages were imported before the install and the session is still '
    'running the earlier code: '
    + ', '.join(f'{n} {v} (expected {EXPECTED[n]})' for n, v in stale.items())
    + '. Restart the session (Runtime > Restart session) and run from the top.'
)
print(f'\nPython {sys.version.split()[0]}; pinned dependency set active')

transformers     4.40.1
tokenizers       0.19.1
timm             0.9.10
huggingface_hub  0.23.4
accelerate       0.30.1

Python 3.12.13; pinned dependency set active


## 1. Mount Drive

In [13]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/v2/bridge'
CONSTRUCTED_DIR = '/content/drive/MyDrive/openvla_cache/v2/constructed'
# One log per generation. The v2 generation harvested its frames fresh and built
# its constructed set fresh, so its predictions live in their own file rather than
# being appended to a log collected over a different scene set.
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv'
print('cache       ->', CACHE_DIR)
print('constructed ->', CONSTRUCTED_DIR)
print('log         ->', PROBE_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache       -> /content/drive/MyDrive/openvla_cache/v2/bridge
constructed -> /content/drive/MyDrive/openvla_cache/v2/constructed
log         -> /content/drive/MyDrive/openvla_cache/v2/probe_predictions.csv


## 2. Import the code

Clones the project code from GitHub into the runtime and imports the loader,
inference, control, and logging functions from there, so the code always matches
the pushed commit.

In [14]:
import sys, os, importlib, subprocess

REPO_URL = 'https://github.com/LewisTL/ECS8056.git'
BRANCH = 'master'
REPO_DIR = '/content/ECS8056'

def sync_repo():
    """Clone or hard-refresh the repository so it matches origin/BRANCH."""
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', url],
                       check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--depth', '1',
                        'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', '--quiet',
                        f'origin/{BRANCH}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch',
                        BRANCH, url, REPO_DIR], check=True)
    return subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD'],
                          capture_output=True, text=True).stdout.strip()


commit = sync_repo()
module_dir = REPO_DIR
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for _m in ('action_bins', 'prediction_log', 'model', 'data', 'controls', 'compose_scenes', 'export_pairs', 'detect_duplicates', 'analysis',):
    sys.modules.pop(_m, None)
importlib.invalidate_caches()

from model import (load_openvla, predict_action, predict_action_dist,
                   describe_action_space, verify_readout, run_metadata,
                   append_prediction_log)
from prediction_log import ensure_readable
from data import (AXIS_INDEX, AXIS_LATERAL, SPLIT_VALIDATION, load_manifest,
                  make_pair, term_axis, term_axis_index)
from controls import (plan_stimuli, build_scene_swap, apply_image_transform,
                      strip_spatial_term, DEFAULT_CONDITIONS)
from compose_scenes import evaluation_scenes, load_constructed_manifest
import analysis
print(f'imported project modules from {module_dir} @ {commit}')

imported project modules from /content/ECS8056 @ fd346f3


## 3. Load OpenVLA-7B

In [15]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-40GB (sm_80, 39.5 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[load_openvla] Loaded. GPU memory allocated: 8.17 GB
{'gpu_name': 'NVIDIA A100-SXM4-40GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.50.1'}


## 4. Action space and readout constants

The continuous readout reimplements the decoding OpenVLA performs inside
`predict_action`, against constants that live in its remote modelling code and
can move between revisions. Those constants are printed here rather than
assumed. The reimplementation is then required to reproduce the executable action
exactly on real inputs before any continuous value is used, which is the gate in
section 7; it runs there rather than here because it draws its inputs from the
probe set, so that what is verified is the stimuli the sweep will actually use.

`bin_width` is the resolution floor of the argmax readout on each dimension: two
predictions closer together than this cannot differ in the executable action, no
matter how differently the model treats them. It is the number the earlier null
results should have been read against, and it sets the equivalence bound in the
analysis notebook.

In [16]:
space = describe_action_space(vla)
for key, value in space.items():
    if isinstance(value, list):
        print(f'{key:20} ' + ' '.join(f'{v:+.5f}' if isinstance(v, float) else str(v)
                                      for v in value))
    else:
        print(f'{key:20} {value}')
print()
print(f"lateral (dx) bin width: {space['bin_width'][0]:.6f}")

unnorm_key           bridge_orig
action_dim           7
n_bins               255
vocab_size           32000
bin_center_first     -0.996078431372549
bin_center_last      0.996078431372549
action_token_id_min  31745
action_token_id_max  31999
q01                  -0.02873 -0.04170 -0.02609 -0.08092 -0.09289 -0.20718 +0.00000
q99                  +0.02831 +0.04086 +0.04016 +0.08192 +0.07793 +0.20383 +1.00000
mask                 True True True True True True False
bin_width            +0.00022 +0.00033 +0.00026 +0.00064 +0.00067 +0.00162 +0.00394

lateral (dx) bin width: 0.000225


## 5. Assemble the probe set

Both scene sources are expanded into the same record shape, so the probe loop
and the log schema do not depend on where a scene came from.

The constructed scenes come from the frozen `evaluation_set.csv`, so exactly the
approved and selected set is measured. The bridge scenes are the frames harvested
into the validation role, which is disjoint from the base frames the constructed
stimuli were built on.

The constructed source is assembled first, and is therefore predicted first. It
carries the experiments while the unaltered frames answer the separate question of
transfer, so a session that ends before the sweep completes should have spent its
GPU time on the experimental set rather than on the secondary one.

For `bridge` scenes the axis comes from the spatial term, and there is no recorded
geometry, so no directional expectation is attached. For `constructed` scenes the
axis is lateral by construction, and three fields carry what construction
recorded: `base_scene_id`, the frame the stimulus was composited from, and the
target sides in image coordinates.

The geometry is logged unconverted. The constant relating image position to the
sign of the lateral action (`IMAGE_X_TO_LATERAL_SIGN`) is the one unresolved link
in the chain, and it is applied in [analysis.py](analysis.py) rather than here, so
that revising it costs a re-read of the log instead of a repeat of every
prediction in it. `base_scene_id` is logged because the frozen set draws each
same-side scene and its `opposite` counterpart from one frame: without it the log
cannot express the pairing, and the decisive contrast falls back to comparing two
groups of scenes when it could hold the frame fixed.

In [17]:
from collections import Counter

probe_set = []

# --- Constructed scenes: the experimental source, as frozen ------------------
# Read from evaluation_set.csv rather than the manifest, so the scenes measured are
# the approved and selected ones and cannot change as construction or screening
# continues. Assembled first so the sweep predicts them first.
constructed = evaluation_scenes(CONSTRUCTED_DIR)
for scene in constructed:
    probe_set.append({
        'scene_source': 'constructed',
        'scene_id': scene['construct_id'],
        'pair_id': scene['construct_id'],
        # The frame this stimulus was composited from, shared with its
        # counterpart arrangement, which is what makes the decisive contrast a
        # within-frame comparison.
        'base_scene_id': str(scene['base_scene_id']),
        'spatial_term': scene['spatial_term'],
        'axis': AXIS_LATERAL,
        'axis_index': AXIS_INDEX[AXIS_LATERAL],
        'configuration': scene['configuration'],
        # Recorded geometry in image coordinates, unconverted: the side of the
        # start position each instruction's own target sits on, and the order of
        # the two. Scoring each instruction against its own target, rather than
        # the pair against their relative order, is what separates scene
        # grounding from a fixed word-to-direction mapping on the same-side
        # arrangements.
        'expected_sign_image': int(scene['expected_sign_image']),
        'target_sign_a_image': int(scene['target_sign_a_image']),
        'target_sign_b_image': int(scene['target_sign_b_image']),
        'category': 'constructed',
        'feasible_both': 'yes',
        'duplicate_target': 'yes',
        'image_path': os.path.join(CONSTRUCTED_DIR, scene['image_path']),
        'instr_a': scene['instr_a'],
        'instr_b': scene['instr_b'],
    })

# --- Bridge scenes: the unaltered validation source --------------------------
# The validation role: unaltered frames, held disjoint from the base frames the
# constructed stimuli were composited from.
bridge_rows = [r for r in load_manifest(CACHE_DIR) if r['split'] == SPLIT_VALIDATION]
for row in bridge_rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    axis_index = term_axis_index(term)
    if axis_index is None:
        # Scene-dependent relations ("closer to the plate") have no axis that can
        # be fixed before seeing the scene, so they carry no expectation.
        continue
    # category_manual overrides the heuristic category when a scene has been
    # manually reviewed; downstream code follows the same fallback.
    category = row.get('category_manual') or row.get('category', 'other')
    probe_set.append({
        'scene_source': 'bridge',
        'scene_id': f"b{int(row['episode_index']):06d}",
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'base_scene_id': '',
        'spatial_term': term,
        'axis': term_axis(term),
        'axis_index': axis_index,
        'configuration': '',
        'expected_sign_image': 0,
        'target_sign_a_image': 0,
        'target_sign_b_image': 0,
        'category': category,
        'feasible_both': row.get('feasible_both', 'unreviewed'),
        'duplicate_target': row.get('duplicate_target', 'unreviewed'),
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

print(f'{len(probe_set)} scenes: ' + str(dict(Counter(
    p['scene_source'] for p in probe_set))))
print('constructed configurations:', dict(Counter(
    p['configuration'] for p in probe_set if p['scene_source'] == 'constructed')))
print('constructed base frames:', len({p['base_scene_id'] for p in probe_set
                                       if p['scene_source'] == 'constructed'}))
print('bridge axes:', dict(Counter(p['axis'] for p in probe_set
                                   if p['scene_source'] == 'bridge')))
if not constructed:
    built = (len(load_constructed_manifest(CONSTRUCTED_DIR))
             if os.path.isfile(os.path.join(CONSTRUCTED_DIR,
                                            'constructed_manifest.csv')) else 0)
    print(f'\nNo frozen evaluation set found ({built} scenes built). Screen and '
          'freeze in Notebook 03 before probing: the experiments run on the frozen '
          'set, and the bridge source alone supports neither the instrument check '
          'the analysis reads nor the decisive comparison.')

156 scenes: {'bridge': 156}
constructed configurations: {}
constructed base frames: 0
bridge axes: {'lateral': 156}

No frozen evaluation set found (612 scenes built). Screen and freeze in Notebook 03 before probing: the experiments run on the frozen set, and the bridge source alone supports neither the instrument check the analysis reads nor the decisive comparison.


## 6. Expand the condition factorial

Every scene is expanded into the predictions its applicable conditions require.
A condition is skipped, not approximated, when its precondition fails: the mirror
conditions need a lateral term because a horizontal flip leaves depth and
vertical relations unchanged, and the term-stripped conditions need a removal
that leaves a well-formed instruction.

Refusals are counted and reported here. A truncated prompt would change the
prediction for reasons unrelated to the spatial term, so producing one would
quietly corrupt the within-scene reference; skipping instead means the neutral
conditions cover a subset of scenes, which the analysis accounts for.

The swapped-scene assignment is a derangement, so no scene is ever paired with
its own image and the control cannot silently degrade into the baseline. It is
built separately per scene source, keeping the replacement image in the same
visual distribution as the original.

For constructed scenes the constraint is stronger than excluding the scene itself.
The frozen set holds two arrangements of the same base frame by design, and a
sibling's image carries the same background, the same object, and a genuine
referent for the instruction this control is meant to strand. Passing
`base_scene_id` as the grouping excludes the whole frame rather than the single
scene, which is what makes the control mean what it says.

In [18]:
swap_map = {}
for source in ('constructed', 'bridge'):
    scenes = [p for p in probe_set if p['scene_source'] == source]
    if len(scenes) < 2:
        continue
    ids = [p['scene_id'] for p in scenes]
    # Constructed scenes are grouped by the frame they were composited from, so a
    # scene never receives an image built from its own frame. Bridge frames stand
    # alone, so the constraint reduces to excluding the scene itself.
    groups = {p['scene_id']: p['base_scene_id'] for p in scenes
              if p['base_scene_id']}
    swap_map.update(build_scene_swap(ids, seed=0, groups=groups))

image_lookup = {p['scene_id']: p['image_path'] for p in probe_set}

work = []
refused = Counter()
for p in probe_set:
    stimuli = plan_stimuli(p, swap_map=swap_map, conditions=DEFAULT_CONDITIONS)
    if strip_spatial_term(p['instr_a'], p['spatial_term']) is None:
        refused[p['scene_source']] += 1
    for stim in stimuli:
        work.append((p, stim))

print(f'{len(work)} predictions across {len(probe_set)} scenes')
print('per condition:', dict(Counter(s.condition for _, s in work)))
print('per scene source:', dict(Counter(p['scene_source'] for p, _ in work)))

# Reported per source rather than pooled. The constructed instructions are
# generated from a fixed template, so a refusal there is a wording fault worth
# seeing, while a refusal on an inherited Bridge instruction is expected.
print('\nterm removal refused (no neutral reference for those scenes):')
for source in ('constructed', 'bridge'):
    total = sum(1 for p in probe_set if p['scene_source'] == source)
    if total:
        print(f"  {source:12} {refused[source]:4}/{total} "
              f"({refused[source] / total:.1%})")

# Both sources appear here, since the sweep runs the constructed scenes first and
# the preview would otherwise show only one of the two record shapes.
print()
for source in ('constructed', 'bridge'):
    for p, stim in [w for w in work if w[0]['scene_source'] == source][:4]:
        print(f"  [{p['scene_id']}] {stim.condition:16} {stim.role} "
              f"{stim.image_transform:14} :: {stim.instruction}")

TypeError: build_scene_swap() got an unexpected keyword argument 'groups'

## 7. Readout correctness gate

Real stimuli are run through both paths and the results compared. Any
disagreement raises, because a continuous value derived from misidentified
constants would be worse than no continuous value at all: it would look plausible
and be wrong.

The sample is drawn from the probe set itself, weighted toward the constructed
scenes, and this matters more than it appears. The continuous readout is a
reimplementation of OpenVLA's decoding, and what needs establishing is that it
agrees on the inputs the experiments are built from. Constructed frames carry a
composited region no unaltered frame contains, so verifying on unaltered frames
alone would leave the experimental stimuli unverified while reporting a pass.

The gate also refuses to pass on a sample too small to be evidence. An empty or
nearly empty sample would otherwise satisfy the equality it checks and let the
sweep proceed on an unverified readout.

In [ ]:
from PIL import Image

GATE_PER_SOURCE = {'constructed': 40, 'bridge': 10}
GATE_MINIMUM = 20  # below this the check is not evidence, whatever it reports

gate_samples = []
gate_counts = Counter()
for source, wanted in GATE_PER_SOURCE.items():
    # Evenly spaced through the source rather than taken from its start, so the
    # sample is not confined to one scene category or one base frame.
    scenes = [p for p in probe_set if p['scene_source'] == source]
    step = max(len(scenes) // wanted, 1)
    for p in scenes[::step][:wanted]:
        gate_samples.append((Image.open(p['image_path']), p['instr_a']))
        gate_counts[source] += 1

assert len(gate_samples) >= GATE_MINIMUM, (
    f'only {len(gate_samples)} stimuli available for the readout gate, which is '
    f'too few to establish agreement. Expected at least {GATE_MINIMUM}; check the '
    f'probe set assembled above.')

gate = verify_readout(processor, vla, gate_samples)
assert gate['matched'] == gate['checked'], gate
print('continuous readout verified against predict_action on '
      f"{gate['checked']} stimuli: {dict(gate_counts)}")

## 8. Start the log

The log is written fresh for this generation. Nothing is migrated from an earlier
one: those predictions were collected over a different scene set, and copying them
forward would put two populations in one file with only a column to tell them
apart, which the analysis would then have to remember to split on everywhere.

The file is created with the complete header from `probe_log_fields()` in
[prediction_log.py](prediction_log.py), the single declaration of the log's
columns, which the probe loop and the analysis notebooks also read. A log whose
header is narrower than the rows later appended to it is not a CSV any reader can
parse, and the failure appears far from its cause: every append succeeds, and the
file breaks only when the analysis first tries to read it.

In [ ]:
import pandas as pd
from prediction_log import PROBE_EXTRA_FIELDS, probe_log_fields

PROBE_LOG_FIELDS = probe_log_fields()
print(f'v4 schema: {len(PROBE_LOG_FIELDS)} columns')

os.makedirs(os.path.dirname(PROBE_CSV), exist_ok=True)
existing = pd.read_csv(PROBE_CSV) if os.path.exists(PROBE_CSV) else None
if existing is not None and len(existing) == 0:
    # An empty file carries a header and nothing else, so replacing it costs
    # nothing and avoids appending under a header declared before the current
    # schema. A log holding rows is left alone: the append path widens it, and
    # rewriting it here would put the two schemas in one file silently.
    existing = None
    print('replacing the empty log with the current header')

if existing is not None:
    print(f'{PROBE_CSV} holds {len(existing)} rows; the probe below resumes into it')
    print('by scene source:', dict(existing['scene_source'].value_counts()))
else:
    pd.DataFrame(columns=PROBE_LOG_FIELDS).to_csv(PROBE_CSV, index=False)
    print(f'created {PROBE_CSV} with the full header')

## 8b. Check the log is readable

A log written before the header was declared in full can hold rows wider than its
own header, which no CSV reader will parse: the error names a line number and
nothing else. `ensure_readable` reports the field counts actually present and
rebuilds the file when they differ, reading each row under the schema matching its
width so that no value moves to a different column.

It refuses to rewrite a log holding rows it cannot account for, since those would
be dropped, so the repair cannot lose data. Nothing is re-run on GPU: this is a
file-format repair, and a no-op on a log that is already consistent.

The analysis notebooks call the same function through `load_inputs`, so the log is
checked wherever it is read rather than only here.

In [ ]:
ensure_readable(PROBE_CSV)

## 9. Instrument-check pilot

The instrument check decides whether the language measurement can mean anything,
so it runs on a small slice before the full sweep rather than after it. With the
instruction held fixed, mirroring the image reverses the lateral axis of the
scene: a model that reads lateral position must change the sign of its lateral
output. If it does not, the visual channel is not live on this axis and a null on
the language comparisons would follow from the model ignoring the image, carrying
no information about spatial language. Learning that from forty scenes rather
than from a completed sweep is the difference between reconsidering the design and
reporting an uninterpretable result.

The slice is drawn from the constructed same-side arrangements, which is where the
check has something to detect. Reflecting an `opposite` arrangement maps the layout
onto itself, so a response near the midpoint gives an antisymmetry and an
invariance both near zero and neither number decides anything. The same-side
arrangements put both instances on one side of the start position, so the
reflection genuinely moves the scene.

Nothing is wasted. The pilot predictions are written to the same log under the
same resume key, so the sweep in the next section skips them.

The runner is defined here and reused by the sweep, so the two cannot drift apart
in what they log. Its resume key is
`(scene_source, pair_id, frame, condition, role, image_scene_id)`. The image
identifier is part of the key because the swapped-scene condition's replacement
image is a function of the pool and the seed: were the pool to change between
sessions, a resumed run would otherwise treat a prediction made against a
different image as already done.

`sample_idx` is fixed at 0 under the deterministic decoding strategy; the column
exists so the schema does not change if repeated sampling is added later.

In [ ]:
import csv
from PIL import Image

FRAME = 'initial'
PILOT_SCENES = 40
PILOT_CONDITIONS = ('baseline', 'mirror', 'neutral', 'mirror_neutral')
PILOT_MIN_FLIP_RATE = 0.5  # below this the lateral channel is not demonstrably live
PILOT_OVERRIDE = False     # set True to sweep anyway, having read the verdict


def resume_key(scene_source, pair_id, condition, role, image_scene_id,
               frame=FRAME):
    """The identity of one prediction, for skipping work already logged."""
    return (scene_source, pair_id, frame, condition, role, image_scene_id)


done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        for r in csv.DictReader(f):
            done.add(resume_key(r.get('scene_source', 'bridge'), r['pair_id'],
                                r.get('condition', 'baseline'), r['role'],
                                r.get('image_scene_id') or r['scene_id'],
                                frame=r.get('frame') or FRAME))
    print(f'resuming: {len(done)} predictions already logged')

image_cache = {}


def load_image(scene_id):
    if scene_id not in image_cache:
        image_cache[scene_id] = Image.open(image_lookup[scene_id]).convert('RGB')
    return image_cache[scene_id]


def run_items(items, label='', report_every=100):
    """Predict and log every item not already in the log, and return the count."""
    ran = 0
    for i, (p, stim) in enumerate(items):
        key = resume_key(p['scene_source'], p['pair_id'], stim.condition,
                         stim.role, stim.image_scene_id)
        if key in done:
            continue
        image = apply_image_transform(stim.image_transform,
                                      load_image(stim.image_scene_id))
        readout = predict_action_dist(processor, vla, image, stim.instruction,
                                      compute_dtype)
        # Keyed by PROBE_EXTRA_FIELDS so the row cannot carry a column the
        # declared header lacks, and asserted rather than trusted.
        extra = {
            'scene_id': p['scene_id'],
            'pair_id': p['pair_id'],
            'base_scene_id': p['base_scene_id'],
            'role': stim.role,
            'frame': FRAME,
            'scene_source': p['scene_source'],
            'condition': stim.condition,
            'image_transform': stim.image_transform,
            'image_scene_id': stim.image_scene_id,
            'configuration': p['configuration'],
            'expected_sign_image': p['expected_sign_image'],
            'target_sign_a_image': p['target_sign_a_image'],
            'target_sign_b_image': p['target_sign_b_image'],
            'spatial_term': p['spatial_term'],
            'axis': p['axis'],
            'axis_index': p['axis_index'],
            'category': p['category'],
            'feasible_both': p['feasible_both'],
            'duplicate_target': p['duplicate_target'],
            'sample_idx': 0,
        }
        assert list(extra) == list(PROBE_EXTRA_FIELDS), (
            'log fields drifted from the schema')
        append_prediction_log(PROBE_CSV, readout.action, stim.instruction, meta,
                              readout=readout, **extra)
        done.add(key)
        ran += 1
        if report_every and ran % report_every == 0:
            print(f'{label}{ran} predictions run '
                  f'({i + 1}/{len(items)} work items seen)')
    return ran


# Evenly spaced through the same-side arrangements, so the slice is not confined
# to one side, one object category, or one stretch of base frames.
same_side = [p for p in probe_set if p['scene_source'] == 'constructed'
             and p['configuration'].startswith('same_side')]
step = max(len(same_side) // PILOT_SCENES, 1)
pilot_scenes = {p['scene_id'] for p in same_side[::step][:PILOT_SCENES]}
pilot = [(p, s) for p, s in work
         if p['scene_id'] in pilot_scenes and s.condition in PILOT_CONDITIONS]

if not pilot:
    print('no constructed same-side scenes are available, so the pilot cannot '
          'run. The instrument check then rests on section 12, after the sweep.')
else:
    print(f'pilot: {len(pilot)} predictions over {len(pilot_scenes)} same-side '
          'scenes')
    run_items(pilot, label='pilot: ', report_every=25)

    pilot_log = pd.read_csv(PROBE_CSV)
    pilot_log = pilot_log[pilot_log['scene_id'].isin(pilot_scenes)
                          & pilot_log['c0'].notna()]
    check = analysis.mirror_check(pilot_log)
    for name in ('neutral', 'term'):
        result = check.get(name, {})
        if not result.get('n'):
            print(f'\n[{name}] no paired mirror predictions')
            continue
        print(f"\n[{name}] n={result['n']}")
        print(f"  sign flips under mirroring        : {result['flip_rate']:.1%}")
        print(f"  identical to original            : {result['identical_rate']:.1%}")
        print(f"  antisymmetry (0 if exact reversal): "
              f"median={result['antisymmetry']['median']:+.5f} "
              f"p={result['antisymmetry']['p_value']:.3g}")
        print(f"  invariance   (0 if ignored)      : "
              f"median={result['invariance']['median']:+.5f} "
              f"p={result['invariance']['p_value']:.3g}")

    # The term-stripped comparison is the cleaner of the two, since it isolates
    # object grounding from any influence of the spatial word, so it decides the
    # verdict where it exists.
    decisive = check.get('neutral') if check.get('neutral', {}).get('n') else \
        check.get('term', {})
    flip_rate = decisive.get('flip_rate', 0.0)
    live = flip_rate >= PILOT_MIN_FLIP_RATE
    print(f"\nverdict: lateral channel {'live' if live else 'not demonstrably live'} "
          f"({flip_rate:.1%} of predictions reverse sign under reflection)")
    assert live or PILOT_OVERRIDE, (
        f'the lateral channel did not reverse on {flip_rate:.1%} of the pilot '
        f'scenes, below the {PILOT_MIN_FLIP_RATE:.0%} this check requires. A null '
        'on the language comparisons would then follow from the image being '
        'ignored rather than from anything about spatial language. Read the '
        'numbers above, then set PILOT_OVERRIDE = True to sweep regardless and '
        'record the finding.')

## 10. Run the probe

The full factorial, run through `run_items` from the previous section. Every work
item the pilot already covered is skipped by the resume key, so nothing is
predicted twice.

The constructed scenes come first, since that is the order the probe set was
assembled in. A session that ends early therefore leaves the experimental source
complete and the validation source partial, rather than the reverse.

In [ ]:
ran = run_items(work, label='sweep: ')
print(f'probe complete: {ran} new predictions -> {PROBE_CSV}')

logged = pd.read_csv(PROBE_CSV)
print('\nrows now in the log, by source and condition:')
print(logged.pivot_table(index='scene_source', columns='condition',
                         values='role', aggfunc='count', fill_value=0))
outstanding = sum(
    1 for p, stim in work
    if resume_key(p['scene_source'], p['pair_id'], stim.condition, stim.role,
                  stim.image_scene_id) not in done)
print(f'\n{outstanding} work items still outstanding')

## 11. Determinism check

Stimuli already in the log are re-predicted. Decoding is greedy with fixed seeds,
so the repeats must agree exactly. Establishing that here means any nonzero
difference in the analysis is attributable to the manipulation rather than to
run-to-run variation, which is what lets differences of the size the continuous
readout resolves be taken seriously at all.

The sample is spread across both scene sources and across conditions rather than
taken from the head of the work list. Taking the first items would draw entirely
from the constructed baseline, leaving the transforms that build a stimulus at run
time (the reflection and the substituted image) unchecked, and those are where a
reproducibility fault would most plausibly sit.

In [ ]:
import numpy as np

REPEAT_PER_CELL = 2  # per (scene source, condition) combination present

# One or two items from each combination of scene source and condition, so every
# transform the sweep applies is re-predicted at least once.
by_cell = {}
for p, stim in work:
    by_cell.setdefault((p['scene_source'], stim.condition), []).append((p, stim))
repeat_items = [item for items in by_cell.values()
                for item in items[:REPEAT_PER_CELL]]
print(f'{len(repeat_items)} stimuli selected across '
      f'{len(by_cell)} source and condition combinations')

repeats = []
for p, stim in repeat_items:
    image = apply_image_transform(stim.image_transform,
                                  load_image(stim.image_scene_id))
    again = predict_action_dist(processor, vla, image, stim.instruction,
                                compute_dtype)
    repeats.append((p['scene_id'], stim.condition, stim.role,
                    again.action, again.expected))

log = pd.read_csv(PROBE_CSV)
worst_action, worst_cont, compared = 0.0, 0.0, 0
for scene_id, condition, role, action, expected in repeats:
    match = log[(log['scene_id'] == scene_id) & (log['condition'] == condition)
                & (log['role'] == role)]
    if match.empty:
        continue
    logged_a = match[[f'a{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    logged_c = match[[f'c{i}' for i in range(7)]].iloc[0].to_numpy(dtype=float)
    worst_action = max(worst_action, float(np.max(np.abs(logged_a - action))))
    if np.isfinite(logged_c).all():
        worst_cont = max(worst_cont, float(np.max(np.abs(logged_c - expected))))
    compared += 1

print(f'{compared} stimuli re-predicted')
print(f'largest argmax difference:     {worst_action:.3e}')
print(f'largest continuous difference: {worst_cont:.3e}')
assert worst_action == 0.0 and worst_cont == 0.0, (
    'repeated identical inputs disagreed; decoding is not deterministic and no '
    'difference measured downstream can be attributed to the manipulation')
print('deterministic')

## 12. Instrument check on the complete log

The pilot verdict, recomputed on everything the sweep produced.

The same comparison as section 9, now with the full sample and reported per scene
source and per arrangement rather than pooled. The strata are not
interchangeable. Reflecting an `opposite` arrangement maps the layout onto itself,
so the check is uninformative there by construction, and pooling it with the
same-side arrangements dilutes the one stratum that carries the signal. The
unaltered Bridge frames form a third stratum, and their result is a separate fact:
it says whether the visual channel is live on the frames the model was trained on
distributions of, which is not established by the constructed frames and does not
establish it.

This check is not itself evidence of spatial language grounding. It establishes
the necessary condition that makes the rest of the analysis interpretable, and it
identifies the lateral axis empirically, which the earlier ground-truth pilot
failed to do.

In [ ]:
log = pd.read_csv(PROBE_CSV)
lateral = log[(log['axis_index'] == AXIS_INDEX[AXIS_LATERAL]) & log['c0'].notna()]


def report(result, heading):
    if not result.get('n'):
        print(f'{heading}: no paired mirror predictions')
        return
    print(f"{heading}  n={result['n']}")
    print(f"  sign flips under mirroring       : {result['flip_rate']:.1%}")
    print(f"  identical to original            : {result['identical_rate']:.1%}")
    print(f"  mean |lateral| original          : {result['mean_abs_original']:.5f}")
    print(f"  mean |change|                    : {result['mean_abs_change']:.5f}")
    print(f"  antisymmetry (0 if exact reversal): "
          f"median={result['antisymmetry']['median']:+.5f} "
          f"p={result['antisymmetry']['p_value']:.3g}")
    print(f"  invariance   (0 if ignored)      : "
          f"median={result['invariance']['median']:+.5f} "
          f"p={result['invariance']['p_value']:.3g}")


for source in ('constructed', 'bridge'):
    subset = lateral[lateral['scene_source'] == source]
    if subset.empty:
        print(f'\n=== {source}: no lateral predictions ===')
        continue
    print(f'\n=== {source} ===')
    per_config = analysis.mirror_check_by_configuration(subset)
    for configuration, result in sorted(per_config.items()):
        for name in ('neutral', 'term'):
            report(result.get(name, {}),
                   f"[{configuration or 'unaltered'} / {name}]")

print('\nRead: on the same-side arrangements, a high flip rate with an '
      'antisymmetry median near zero means the lateral channel tracks the scene '
      'and the language comparisons are interpretable. A near-zero flip rate with '
      'an invariance median near zero means the output ignores the image, and no '
      'language conclusion can be drawn from this axis. The opposite arrangement '
      'is close to symmetric under reflection, so a small change there is '
      'expected and decides nothing either way.')